# DeiT-III-style baseline on Kaggle CIFAR-LT and ImageNet-LT

The supplied custom LayerScale / DropPath transformer with 3-Augment and BCE loss. This is a DeiT-III-style baseline, separate from the DeiT-LT method in the attached paper.


## Datasets covered

The supplied paper, **DeiT-LT: Distillation Strikes Back for Vision Transformer Training on Long-Tailed Datasets**
(arXiv:2404.02900v1), evaluates these four benchmarks (Section 4.1 and Supplement A.1):

| Benchmark | Classes | Training distribution | Experiments here |
|---|---:|---|---|
| CIFAR-10-LT | 10 | Exponential sampling from balanced CIFAR-10 | IF=100 and IF=50 |
| CIFAR-100-LT | 100 | Exponential sampling from balanced CIFAR-100 | IF=100 and IF=50 |
| ImageNet-LT | 1,000 | Official LT training manifest, unchanged | Official split |
| iNaturalist-2018 | 8,142 | Natural class imbalance, unchanged | Loader available, skipped by request |

The paper reports 115,846 ImageNet-LT training images and 437,513 iNaturalist-2018 training images.
At IF=100 the standard CIFAR-LT construction gives 12,406 and 10,847 training images respectively.
Balanced CIFAR test sets remain untouched. Dataset references such as ImageNet-21K and CLIP's
400M image-text pairs concern other methods' pretraining. They are not DeiT-LT evaluation benchmarks.
OOD distillation in DeiT-LT uses augmented training images, not a separate OOD dataset.

### Kaggle setup

1. Create a Kaggle notebook, enable a GPU, and use **Add Input** to attach these verified existing sources:
   - CIFAR-10: [pankrzysiu/cifar10-python](https://www.kaggle.com/datasets/pankrzysiu/cifar10-python)
   - CIFAR-100: [fedesoriano/cifar100](https://www.kaggle.com/datasets/fedesoriano/cifar100)
   - ImageNet-LT: [chanplusplus/imagenetlt](https://www.kaggle.com/datasets/chanplusplus/imagenetlt)
   Their public file listings were inspected on 23 October 2026. No bulk image download to your computer is necessary.
   **iNaturalist-2018 is skipped in the default runs, as requested. Its loader remains available.**
   To enable it later, supply its correct input path in `DATASET_PATHS` and append it to `EXPERIMENTS`.
   No complete public Kaggle image mirror was verified. The similarly named
   `authuria/inaturalist` upload contains the 2021 mini dataset and is not a replacement for this benchmark.
2. Run this notebook from the top. It requires the standard Kaggle Python image with PyTorch,
   torchvision, NumPy, pandas, SciPy, Pillow and matplotlib. It makes no network requests or dataset downloads.
3. `INPUT_ROOT` defaults to `/kaggle/input`. It discovers files within five directory levels.
   If multiple copies exist or the layout differs, fill `DATASET_PATHS` in the configuration cell.
4. Outputs go to `/kaggle/working/long_tail_results`, under separate recipe / experiment / model folders.
   Save a Kaggle notebook version to retain those outputs. An interrupted run resumes from `last.pt`
   when its settings match. For a new experiment budget or changed data, change the output directory
   or set `RESUME=False`. Checkpoints are local trusted files created by this notebook.
5. The default registry contains **five experiment settings** for the notebook's models, at 100 epochs:
   CIFAR-10-LT and CIFAR-100-LT at IF=100/50, plus ImageNet-LT. The chosen datasets must all be attached.
   iNaturalist-2018 does not run. Setting `ALLOW_MISSING_DATASETS=True` explicitly permits running a subset
   of the selected experiments, with missing entries recorded in `missing_datasets.csv`.
   This is a substantial GPU workload. Adjust `EPOCHS`, `BATCH_SIZE`, `NUM_WORKERS` or `EXPERIMENTS`
   before running if you want a shorter trial. `LT_SMOKE_TEST=1` is a developer diagnostic that uses
   reduced models and partial batches. It must not be used to report benchmark accuracy.

Supported extracted layouts (archives must already be extracted):

```text
/kaggle/input/<bundle>/
  cifar-10-batches-py/          # data_batch_1..5, test_batch
  cifar-100-python/            # train, test
  imagenet_lt/
    ImageNet_LT_train.txt      # relative/path.jpg integer_label
    ImageNet_LT_val.txt        # optional separate selection validation
    ImageNet_LT_test.txt       # labeled final evaluation
    train/<class>/...jpg
    val/<class>/...jpg
  inaturalist2018/
    train2018.json             # COCO-style images, annotations, categories
    val2018.json               # labeled final evaluation
    train_val2018/...jpg
```

The verified ImageNet-LT source stores renamed PNGs below
`/kaggle/input/imagenetlt/torch_image_folder/mnt/volume_sfo3_01/imagenet-lt/ImageDataset`.
The configuration detects both `/kaggle/input/imagenetlt` and the owner-qualified mount
`/kaggle/input/datasets/chanplusplus/imagenetlt` (and equivalent mount variants for both CIFAR sources).
It reads the numeric class folders and checks each split's
per-class counts against the supplied official manifests. It preserves numeric class IDs rather than sorting
their names alphabetically. The source also contains TFRecords, which this notebook does not need.
The Kaggle CIFAR-100 input stores `train`, `test` and `meta` directly under `/kaggle/input/cifar100`.
These verified presets take priority only when those directories exist.

Image roots and manifests may live in separate bundle subdirectories. ImageNet's image folder
can also be named `imagenet`. `image_root` overrides image resolution independently of manifest location.
All datasets also support named `*_train.txt` / `*_val.txt` / `*_test.txt` manifests. A dataset directory
with `train/<class>/` and `test/<class>/` (optionally `val/<class>/`) is supported too. Folder label mappings
come from the training folders and are reused for evaluation. For ImageNet-LT, the folder-format training
set must already contain the official LT subset. Do not substitute the balanced ImageNet training set.
For CIFAR, supply balanced originals for both ratios, or provide matching pre-made LT splits when running
one ratio at a time. The loader refuses to imbalance an already imbalanced split twice.
An unlabeled iNaturalist competition test JSON is ignored automatically. A lone labeled `val` split
serves as final evaluation and never drives checkpoint selection.

### Models, grouping and evaluation

These notebooks extend the supplied baseline implementations across the paper's datasets.
They **do not implement DeiT-LT**, its DIST token, SAM teacher, DRW, Mixup/CutMix distillation,
or the paper's 1,200 / 1,400 / 1,000-epoch training schedules. They do not claim to reproduce its scores.
The custom transformer keeps the supplied 192-dimensional, 8-block architecture. It uses 224-pixel inputs
and 16-pixel patches by default. The CIFAR CNN uses 32-pixel inputs. The ImageNet/iNat CNN is ResNet-50
trained from scratch, matching the CNN family used for large datasets in the paper, without its teacher recipe.

CIFAR groups follow Supplement A.1: CIFAR-10 uses class IDs 0–2 / 3–6 / 7–9, and CIFAR-100 uses
0–35 / 36–70 / 71–99. The same fixed groups are used for both imbalance ratios. This assumes the
standard class order used for exponential sampling. ImageNet-LT/iNat groups use training counts
>100 / 20–100 / <20. Head/mid/tail accuracy means average per-class accuracy within each group.

If a separate validation split exists, it selects the best checkpoint. Otherwise the final epoch is evaluated.
The final evaluation data never selects the checkpoint. Metrics include top-1, top-5, macro/weighted F1,
macro precision/recall, class-group accuracy and per-class accuracy. Evaluation streams counts without keeping
all logits in memory. Large confusion matrices remain sparse. Scores are fractions from 0 to 1.
Missing evaluation support produces undefined per-class/group accuracies rather than invented zeros.

Paper source: supplied `2404.02900v1-1.pdf`, Section 4.1, Supplement A.1–A.3 and Table S.1.


## 1. Configuration

Edit paths or training budget here before running the remaining cells.

In [1]:
import os, json, random, time, gc, math, copy, pickle, warnings
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision
import torchvision.transforms as T
from scipy import sparse

# All input data are local. No downloads or pretrained weights are requested.
INPUT_ROOT = Path(os.environ.get('LT_INPUT_ROOT', '/kaggle/input'))
OUTPUT_ROOT = Path(os.environ.get('LT_OUTPUT_ROOT', '/kaggle/working/long_tail_results'))
SMOKE_TEST = os.environ.get('LT_SMOKE_TEST', '0') == '1'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED = 42
EPOCHS = 1 if SMOKE_TEST else 100  # baseline budget, not the DeiT-LT paper schedule
BATCH_SIZE = 2 if SMOKE_TEST else 64
NUM_WORKERS = 0 if SMOKE_TEST else min(4, os.cpu_count() or 1)
USE_AMP = DEVICE.type == 'cuda'
RESUME = True
CHECKPOINT_EVERY = 1
MAX_TRAIN_BATCHES = 1 if SMOKE_TEST else None
MAX_EVAL_BATCHES = 2 if SMOKE_TEST else None
VIT_IMAGE_SIZE = 32 if SMOKE_TEST else 224
VIT_PATCH_SIZE = 4 if SMOKE_TEST else 16
LARGE_CNN_IMAGE_SIZE = 32 if SMOKE_TEST else 224

# Each entry can override root, image_root, train, val and test paths.
# Example: DATASET_PATHS['imagenet_lt'] = {
#   'root': '/kaggle/input/my-bundle/imagenet_lt',
#   'image_root': '/kaggle/input/my-bundle/imagenet',
#   'train': '/kaggle/input/my-bundle/splits/ImageNet_LT_train.txt',
#   'val': '/kaggle/input/my-bundle/splits/ImageNet_LT_val.txt',
#   'test': '/kaggle/input/my-bundle/splits/ImageNet_LT_test.txt'}
DATASET_PATHS = {'cifar10_lt': {}, 'cifar100_lt': {},
                 'imagenet_lt': {}, 'inaturalist2018': {}}
KAGGLE_SOURCES = {
    'cifar10_lt': 'pankrzysiu/cifar10-python',
    'cifar100_lt': 'fedesoriano/cifar100',
    'imagenet_lt': 'chanplusplus/imagenetlt',
    'inaturalist2018': None,  # No complete public Kaggle mirror verified.
}
# Attach the three sources above with Kaggle's Add Input control.
# Defaults apply only when the verified source layout is mounted.
_source_suffixes = {
    'cifar10_lt': Path('cifar-10-batches-py'),
    'cifar100_lt': Path('.'),
    'imagenet_lt': Path('torch_image_folder/mnt/volume_sfo3_01/imagenet-lt/ImageDataset'),
}
for _key, _ref in KAGGLE_SOURCES.items():
    if not _ref:
        continue
    _owner, _slug = _ref.split('/')
    # Kaggle supports both legacy flat mounts and owner-qualified dataset mounts.
    _mounts = [INPUT_ROOT / _slug, INPUT_ROOT / 'datasets' / _owner / _slug,
               INPUT_ROOT / _owner / _slug]
    for _mount in _mounts:
        _root = _mount / _source_suffixes[_key]
        if _root.is_dir():
            DATASET_PATHS[_key]['root'] = str(_root)
            if _key == 'imagenet_lt':
                DATASET_PATHS[_key].update(prefer_folders=True, manifest_root=str(_mount))
            break

# Require the three chosen benchmarks. iNaturalist-2018 is excluded at the user's request.
ALLOW_MISSING_DATASETS = False
DATASET_META = {
    'cifar10_lt': {'classes': 10, 'title': 'CIFAR-10-LT'},
    'cifar100_lt': {'classes': 100, 'title': 'CIFAR-100-LT'},
    'imagenet_lt': {'classes': 1000, 'title': 'ImageNet-LT'},
    'inaturalist2018': {'classes': 8142, 'title': 'iNaturalist-2018'},
}
# The paper evaluates both CIFAR datasets at rho=100 and rho=50.
EXPERIMENTS = [
    {'dataset': 'cifar10_lt', 'imbalance_ratio': 100},
    {'dataset': 'cifar10_lt', 'imbalance_ratio': 50},
    {'dataset': 'cifar100_lt', 'imbalance_ratio': 100},
    {'dataset': 'cifar100_lt', 'imbalance_ratio': 50},
    {'dataset': 'imagenet_lt', 'imbalance_ratio': None},
]
# To add iNaturalist-2018 later, attach its correct 2018 image data, fill
# DATASET_PATHS['inaturalist2018'], and append:
# EXPERIMENTS.append({'dataset': 'inaturalist2018', 'imbalance_ratio': None})

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()
if SMOKE_TEST:
    torch.set_num_threads(2)
torch.backends.cudnn.benchmark = False
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print({'device': str(DEVICE), 'torch': torch.__version__,
       'torchvision': torchvision.__version__, 'input_root': str(INPUT_ROOT),
       'experiments': len(EXPERIMENTS), 'smoke_test': SMOKE_TEST})


{'device': 'cuda', 'torch': '2.11.0+cu128', 'torchvision': '0.26.0+cu128', 'input_root': '/kaggle/input', 'experiments': 5, 'smoke_test': False}


In [2]:
NOTEBOOK_RECIPE = 'deit3'
MODEL_NAMES = ['deit3']

## 2. Dataset discovery, loading and preflight

Attached benchmarks must pass validation before training. Missing benchmarks appear in an explicit report.

In [3]:
# Discovery searches metadata directories, not millions of individual image paths.
ALIASES = {
    'cifar10_lt': ['cifar10', 'cifar-10', 'cifar10_lt', 'cifar-10-lt'],
    'cifar100_lt': ['cifar100', 'cifar-100', 'cifar100_lt', 'cifar-100-lt'],
    'imagenet_lt': ['imagenet_lt', 'imagenet-lt', 'imagenetlt'],
    'inaturalist2018': ['inaturalist2018', 'inaturalist-2018', 'inaturalist18', 'inat2018', 'inat18', 'inat-2018'],
}

def normalized(value):
    return ''.join(ch for ch in str(value).lower() if ch.isalnum())

def metadata_inventory(root, max_depth=5):
    root = Path(root)
    if not root.is_dir():
        raise FileNotFoundError(f'Input directory does not exist: {root}. Attach the Kaggle dataset or set INPUT_ROOT.')
    directories, files = [root], []
    # Keep train/val/test parents for folder-format datasets, but avoid their images.
    image_dirs = {'images', 'imgs', 'jpegimages', 'train_val2018', 'train', 'val', 'test'}
    for current, dirs, names in os.walk(root):
        current = Path(current)
        depth = len(current.relative_to(root).parts)
        if depth >= max_depth or (depth > 0 and current.name.lower() in image_dirs):
            dirs[:] = []
        else:
            dirs[:] = sorted(d for d in dirs if not d.startswith('.'))
        directories.extend(current / d for d in dirs)
        files.extend(current / name for name in sorted(names)
                     if name.lower().endswith(('.txt', '.json'))
                     or name in {'data_batch_1', 'train', 'test_batch', 'meta'})
    return sorted(set(directories)), sorted(files)

DIRECTORIES, METADATA_FILES = metadata_inventory(INPUT_ROOT)

def choose_unique(paths, description):
    paths = sorted(set(Path(p) for p in paths))
    if len(paths) > 1:
        raise ValueError(f'Ambiguous {description}: {[str(p) for p in paths]}. Set DATASET_PATHS explicitly.')
    return paths[0] if paths else None

def find_named_file(key, split, root=None):
    override = DATASET_PATHS[key].get(split)
    if override:
        path = Path(override)
        if not path.is_file():
            raise FileNotFoundError(path)
        return path
    named = []
    for alias in ALIASES[key]:
        for ext in ['txt', 'json']:
            named.extend([normalized(f'{alias}_{split}.{ext}'),
                          normalized(f'{split}_{alias}.{ext}')])
    if key == 'inaturalist2018':
        named.extend([normalized(f'{split}2018.json'), normalized(f'{split}2018.txt')])
    if root is not None:
        # Read explicit roots directly as they may sit beyond the inventory depth
        # or be mounted through symlinks that os.walk deliberately does not follow.
        root = Path(root)
        local = [p for p in root.iterdir() if p.is_file()
                 and (normalized(p.name) in named
                      or p.name.lower() in {f'{split}.txt', f'{split}.json'})]
        if local:
            return choose_unique(local, f'{key} {split} manifest in {root}')
    candidates = [p for p in METADATA_FILES if normalized(p.name) in named]
    if root is not None:
        candidates.extend(p for p in METADATA_FILES if p.parent == root
                          and p.name.lower() in {f'{split}.txt', f'{split}.json'})
    return choose_unique(candidates, f'{key} {split} manifest')

def find_dataset_root(key):
    override = DATASET_PATHS[key].get('root')
    if override:
        root = Path(override)
        if not root.is_dir():
            raise FileNotFoundError(root)
        return root
    candidates = [d for d in DIRECTORIES if normalized(d.name) in
                  {normalized(a) for a in ALIASES[key]}]
    if candidates:
        return choose_unique(candidates, f'{key} root')
    return None

class ArrayImages(Dataset):
    def __init__(self, data, targets):
        self.data = data
        self.targets = np.asarray(targets, dtype=np.int64)
    def __len__(self):
        return len(self.targets)
    def __getitem__(self, index):
        return Image.fromarray(self.data[index]).convert('RGB'), int(self.targets[index])

class FileImages(Dataset):
    def __init__(self, paths, targets):
        self.paths = list(map(Path, paths))
        self.targets = np.asarray(targets, dtype=np.int64)
    def __len__(self):
        return len(self.targets)
    def __getitem__(self, index):
        try:
            with Image.open(self.paths[index]) as image:
                return image.convert('RGB'), int(self.targets[index])
        except Exception as exc:
            raise RuntimeError(f'Cannot decode image {self.paths[index]}') from exc

class TransformedSubset(Dataset):
    def __init__(self, base, indices, transform):
        self.base, self.indices, self.transform = base, np.asarray(indices), transform
        self.targets = np.asarray(base.targets)[self.indices]
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, index):
        image, label = self.base[int(self.indices[index])]
        return self.transform(image), label

def read_cifar_raw(key):
    # Official extracted Python archives, loaded without downloading or writing to /kaggle/input.
    cname = 'cifar-10-batches-py' if key == 'cifar10_lt' else 'cifar-100-python'
    override = find_dataset_root(key)
    candidates = [d for d in DIRECTORIES if d.name == cname]
    if override:
        candidates = [d for d in [override, override / cname]
                      if (d / ('data_batch_1' if key == 'cifar10_lt' else 'train')).is_file()]
    root = choose_unique(candidates, f'{key} extracted Python archive')
    if root is None:
        return None
    def batch(names):
        arrays, labels = [], []
        for name in names:
            with open(root / name, 'rb') as handle:
                obj = pickle.load(handle, encoding='latin1')
            arrays.append(np.asarray(obj['data']).reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1))
            labels.extend(obj['labels'] if key == 'cifar10_lt' else obj['fine_labels'])
        return ArrayImages(np.concatenate(arrays), labels)
    train_names = [f'data_batch_{i}' for i in range(1, 6)] if key == 'cifar10_lt' else ['train']
    test_names = ['test_batch'] if key == 'cifar10_lt' else ['test']
    return batch(train_names), None, batch(test_names), {'format': 'official-cifar-python', 'root': str(root)}

def manifest_entries(path, category_map=None):
    if path.suffix.lower() == '.json':
        with open(path) as handle:
            obj = json.load(handle)
        if not isinstance(obj, dict) or 'images' not in obj or 'annotations' not in obj:
            raise ValueError(f'{path}: expected COCO/iNaturalist images and annotations arrays.')
        if category_map is None:
            ids = sorted(c['id'] for c in obj.get('categories', []))
            if not ids:
                ids = sorted({a['category_id'] for a in obj['annotations']})
            category_map = {category: i for i, category in enumerate(ids)}
        labels = {}
        for ann in obj['annotations']:
            image_id, category = ann['image_id'], ann['category_id']
            if category not in category_map:
                raise ValueError(f'Unknown category {category} in {path}')
            if image_id in labels and labels[image_id] != category_map[category]:
                raise ValueError(f'Conflicting labels for image {image_id} in {path}')
            labels[image_id] = category_map[category]
        entries = [(i['file_name'], labels[i['id']]) for i in obj['images'] if i['id'] in labels]
        if len(entries) != len(obj['images']):
            raise ValueError(f'{path}: unlabeled images cannot be used for evaluation.')
        return entries, category_map
    entries = []
    for line_number, line in enumerate(path.read_text().splitlines(), 1):
        line = line.strip()
        if not line or line.startswith('#'):
            continue
        try:
            filename, label = line.rsplit(None, 1)
            entries.append((filename, int(label)))
        except Exception as exc:
            raise ValueError(f'{path}:{line_number}: expected relative/image/path label') from exc
    return entries, category_map

def resolve_manifest_images(key, manifest, entries, dataset_root):
    if not entries:
        raise ValueError(f'Empty manifest: {manifest}')
    configured = DATASET_PATHS[key].get('image_root')
    roots = [Path(configured)] if configured else []
    if not configured:
        roots += [dataset_root, manifest.parent, manifest.parent.parent, INPUT_ROOT]
        roots += [d for d in DIRECTORIES if normalized(d.name) in
                  {'images', 'imgs', 'imagenet', 'imagenet1k', 'ilsvrc2012', 'trainval2018',
                   'inaturalist2018', 'inat2018'}]
        for parent in [dataset_root, manifest.parent, manifest.parent.parent]:
            if parent:
                roots.extend(parent / child for child in ['images', 'data', 'train_val2018'])
    roots = list(dict.fromkeys(Path(p) for p in roots if p is not None))
    sample = entries[:min(16, len(entries))]
    # Some bundles add/remove an ImageNet/ or iNaturalist/ prefix.
    for trim in range(4):
        for root in roots:
            def location(filename):
                p = Path(filename.replace('\\', '/'))
                if p.is_absolute():
                    return p
                return root.joinpath(*p.parts[trim:])
            if all(location(name).is_file() for name, _ in sample):
                return [location(name) for name, _ in entries], {'image_root': str(root), 'trim_components': trim}
    raise FileNotFoundError(f'Cannot resolve images in {manifest}. First path: {entries[0][0]}. '
                            f'Set DATASET_PATHS[{key!r}]["image_root"] to the extracted image directory.')

def folder_dataset(path, class_to_idx=None):
    ds = torchvision.datasets.ImageFolder(str(path))
    if class_to_idx is None:
        # Numeric folder names carry the original class IDs. Lexical ordering
        # would map 10 to class 2 and corrupt comparisons with official manifests.
        if all(name.isdigit() for name in ds.classes):
            class_to_idx = {name: int(name) for name in ds.classes}
        else:
            class_to_idx = ds.class_to_idx
    elif not set(ds.classes) <= set(class_to_idx):
        raise ValueError(f'{path}: evaluation has unknown class folders.')
    paths = [p for p, _ in ds.samples]
    targets = [class_to_idx[ds.classes[label]] for _, label in ds.samples]
    return FileImages(paths, targets), class_to_idx

def read_verified_kaggle_folders(key, root):
    train, mapping = folder_dataset(root / 'train')
    val = folder_dataset(root / 'val', mapping)[0] if (root / 'val').is_dir() else None
    test = folder_dataset(root / 'test', mapping)[0] if (root / 'test').is_dir() else None
    if test is None:
        raise FileNotFoundError(f'{root}: missing final test class folders.')
    manifest_root = Path(DATASET_PATHS[key]['manifest_root'])
    checks = {}
    for split, dataset in [('train', train), ('val', val), ('test', test)]:
        if dataset is None:
            continue
        manifest = find_named_file(key, split, manifest_root)
        if manifest is None:
            raise FileNotFoundError(f'{key}: missing {split} manifest for Kaggle folder verification.')
        entries, _ = manifest_entries(manifest)
        expected = Counter(label for _, label in entries)
        actual = Counter(map(int, dataset.targets))
        if expected != actual:
            raise ValueError(f'{key}/{split}: numeric class folders disagree with official manifest class counts.')
        checks[split] = {'manifest': str(manifest), 'images': len(dataset), 'per_class_counts_match': True}
    return train, val, test, {'format': 'kaggle-numeric-class-folders', 'root': str(root),
                              'source': KAGGLE_SOURCES[key], 'manifest_checks': checks}

def read_dataset_sources(key):
    if key.startswith('cifar'):
        raw = read_cifar_raw(key)
        if raw:
            return raw
    root = find_dataset_root(key)
    if key == 'imagenet_lt' and root and not DATASET_PATHS[key].get('image_root'):
        # Detect the verified renamed-image layout independently of the config
        # presets, including when a user supplies the dataset mount as root.
        folder_root = root / 'torch_image_folder/mnt/volume_sfo3_01/imagenet-lt/ImageDataset'
        if folder_root.is_dir():
            DATASET_PATHS[key].update(root=str(folder_root), prefer_folders=True,
                                     manifest_root=str(root))
            root = folder_root
    if root and DATASET_PATHS[key].get('prefer_folders'):
        return read_verified_kaggle_folders(key, root)
    train_file = find_named_file(key, 'train', root)
    val_file = find_named_file(key, 'val', root)
    test_file = find_named_file(key, 'test', root)
    if key == 'inaturalist2018' and test_file and not DATASET_PATHS[key].get('test'):
        # The public competition test JSON is often included but has no labels.
        # Use the official labeled val2018 split as final evaluation instead.
        if test_file.suffix.lower() == '.json':
            with open(test_file) as handle:
                test_object = json.load(handle)
            if not test_object.get('annotations'):
                test_file = None
    if train_file:
        source, mapping, details = {}, None, {'format': 'manifest', 'manifests': {}}
        for split, path in [('train', train_file), ('val', val_file), ('test', test_file)]:
            if path is None:
                source[split] = None
                continue
            entries, mapping = manifest_entries(path, mapping)
            paths, layout = resolve_manifest_images(key, path, entries, root)
            source[split] = FileImages(paths, [label for _, label in entries])
            details['manifests'][split] = {'path': str(path), **layout}
        if source['test'] is None:
            # iNat 2018 has a labeled validation set and an unlabeled competition test set.
            # A lone labeled val split is final evaluation, never checkpoint selection.
            source['test'], source['val'] = source['val'], None
        if source['test'] is None:
            raise FileNotFoundError(f'{key}: a labeled test/val manifest is required.')
        if source['val'] is not None:
            val_paths, test_paths = set(source['val'].paths), set(source['test'].paths)
            if val_paths & test_paths:
                raise ValueError(f'{key}: validation and test images overlap.')
        for split in ['val', 'test']:
            if source[split] is not None and set(source['train'].paths) & set(source[split].paths):
                raise ValueError(f'{key}: train and {split} images overlap.')
        return source['train'], source['val'], source['test'], details
    if root and (root / 'train').is_dir():
        train, mapping = folder_dataset(root / 'train')
        val = folder_dataset(root / 'val', mapping)[0] if (root / 'val').is_dir() else None
        test = folder_dataset(root / 'test', mapping)[0] if (root / 'test').is_dir() else None
        if test is None:
            test, val = val, None
        if test is not None:
            return train, val, test, {'format': 'class-folders', 'root': str(root)}
    raise FileNotFoundError(f'{key}: no supported data found. Provide extracted CIFAR Python files, '
                            'labeled manifests, or dataset/train/<class> and dataset/test/<class> folders. '
                            'ImageNet-LT requires the official LT split, not balanced ImageNet training data.')

def long_tail_indices(targets, num_classes, imbalance_ratio, seed=SEED):
    labels = np.asarray(targets)
    counts = np.bincount(labels, minlength=num_classes)
    if np.any(counts == 0):
        raise ValueError('Every training class must be present before long-tail sampling.')
    if counts.max() != counts.min():
        # Never silently long-tail an already imbalanced dataset a second time.
        observed = counts.max() / counts.min()
        if not np.isclose(observed, imbalance_ratio, rtol=0.05):
            raise ValueError(f'Already imbalanced CIFAR data have rho={observed:.2f}, requested {imbalance_ratio}. '
                             'Supply balanced CIFAR or a matching pre-made LT split.')
        return np.arange(len(labels)), counts, 'pre-existing LT split'
    sizes = np.array([max(1, int(counts.max() * (1 / imbalance_ratio) ** (c / (num_classes - 1))))
                      for c in range(num_classes)])
    rng = np.random.default_rng(seed)
    selected = np.concatenate([rng.permutation(np.flatnonzero(labels == c))[:sizes[c]]
                               for c in range(num_classes)])
    rng.shuffle(selected)
    return selected, sizes, 'exponential sampling from balanced CIFAR'

def shot_groups(key, counts):
    if key == 'cifar10_lt':
        # Supplement A.1: first 3 / next 4 / last 3 classes.
        return {'many': np.arange(0, 3), 'medium': np.arange(3, 7), 'few': np.arange(7, 10)}
    if key == 'cifar100_lt':
        # Supplement A.1: first 36 / next 35 / final 29 classes.
        return {'many': np.arange(0, 36), 'medium': np.arange(36, 71), 'few': np.arange(71, 100)}
    return {'many': np.flatnonzero(counts > 100),
            'medium': np.flatnonzero((counts >= 20) & (counts <= 100)),
            'few': np.flatnonzero(counts < 20)}

def prepare_experiment(experiment):
    key, ratio = experiment['dataset'], experiment['imbalance_ratio']
    nc = DATASET_META[key]['classes']
    train, val, test, details = read_dataset_sources(key)
    for split, dataset in [('train', train), ('val', val), ('test', test)]:
        if dataset is not None:
            labels = np.asarray(dataset.targets)
            if len(labels) == 0 or labels.min() < 0 or labels.max() >= nc:
                raise ValueError(f'{key}/{split}: labels must be 0..{nc-1}, and the split must be nonempty.')
    if key.startswith('cifar'):
        indices, counts, sampling = long_tail_indices(train.targets, nc, ratio)
    else:
        indices = np.arange(len(train))
        counts = np.bincount(train.targets, minlength=nc)
        sampling = 'official long-tail split, unchanged'
        if not SMOKE_TEST and (np.any(counts == 0) or counts.max() == counts.min()):
            raise ValueError(f'{key}: expected a long-tailed training split covering all {nc} classes.')
        expected_size = {'imagenet_lt': 115846, 'inaturalist2018': 437513}[key]
        if not SMOKE_TEST and len(train) != expected_size:
            warnings.warn(f'{key}: found {len(train):,} training images; the paper reports '
                          f'{expected_size:,}. Verify that the supplied split is the intended benchmark.')
    if not SMOKE_TEST and np.unique(test.targets).size != nc:
        raise ValueError(f'{key}: labeled evaluation must cover all {nc} classes.')
    tag = key + (f'_if{ratio}' if ratio else '')
    return {'key': key, 'tag': tag, 'num_classes': nc, 'train': train, 'val': val, 'test': test,
            'train_indices': indices, 'counts': counts, 'groups': shot_groups(key, counts),
            'details': {**details, 'sampling': sampling, 'imbalance_ratio': ratio}}

# Fail early before starting a long run if any benchmark is missing.
def preflight():
    global ACTIVE_EXPERIMENTS
    rows, missing = [], []
    ACTIVE_EXPERIMENTS = []
    for exp in EXPERIMENTS:
        try:
            data = prepare_experiment(exp)
        except FileNotFoundError as exc:
            if not ALLOW_MISSING_DATASETS:
                raise
            missing.append({'dataset': exp['dataset'], 'imbalance_ratio': exp['imbalance_ratio'],
                            'reason': str(exc)})
            continue
        for base, indices in [(data['train'], data['train_indices']),
                              (data['test'], np.arange(len(data['test'])))]:
            for i in indices[np.linspace(0, len(indices)-1, min(8, len(indices)), dtype=int)]:
                image, label = base[int(i)]
                assert image.mode == 'RGB' and 0 <= label < data['num_classes']
        rows.append({'experiment': data['tag'], 'classes': data['num_classes'],
                     'train': len(data['train_indices']),
                     'selection_val': len(data['val']) if data['val'] is not None else 0,
                     'final_eval': len(data['test']), 'head_max': int(data['counts'].max()),
                     'tail_min': int(data['counts'].min()), 'format': data['details']['format']})
        ACTIVE_EXPERIMENTS.append(exp)
        del data
        gc.collect()
    if missing:
        print('MISSING BENCHMARKS: these will not run and no results will be reported for them.')
        display(pd.DataFrame(missing))
        pd.DataFrame(missing).to_csv(OUTPUT_ROOT / 'missing_datasets.csv', index=False)
    else:
        (OUTPUT_ROOT / 'missing_datasets.csv').unlink(missing_ok=True)
    if not ACTIVE_EXPERIMENTS:
        raise FileNotFoundError('No requested dataset is attached. Add the Kaggle sources listed in KAGGLE_SOURCES.')
    return pd.DataFrame(rows)

dataset_summary = preflight()
display(dataset_summary)
dataset_summary.to_csv(OUTPUT_ROOT / 'dataset_inventory.csv', index=False)


,experiment,classes,train,selection_val,final_eval,head_max,tail_min,format
0,cifar10_lt_if100,10,12406,0,10000,5000,50,official-cifar-python
1,cifar10_lt_if50,10,13996,0,10000,5000,100,official-cifar-python
2,cifar100_lt_if100,100,10847,0,10000,500,5,official-cifar-python
3,cifar100_lt_if50,100,12608,0,10000,500,10,official-cifar-python
4,imagenet_lt,1000,115846,0,50000,1280,5,kaggle-numeric-class-folders


## 3. Model definitions

In [4]:
class BasicBlock(nn.Module):
    expansion = 1
    def __init__(self, in_planes, planes, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, planes, 1, stride, bias=False),
                nn.BatchNorm2d(planes))
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        return F.relu(out)

class ResNetCIFAR(nn.Module):
    # ResNet-32 = 3 stages x 5 blocks, He et al. CIFAR variant
    def __init__(self, num_blocks=(5,5,5), num_classes=100):
        super().__init__()
        self.in_planes = 16
        self.conv1 = nn.Conv2d(3, 16, 3, 1, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(16)
        self.layer1 = self._make_layer(16, num_blocks[0], 1)
        self.layer2 = self._make_layer(32, num_blocks[1], 2)
        self.layer3 = self._make_layer(64, num_blocks[2], 2)
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(64, num_classes)
    def _make_layer(self, planes, n, stride):
        strides = [stride] + [1]*(n-1)
        layers = []
        for s in strides:
            layers.append(BasicBlock(self.in_planes, planes, s))
            self.in_planes = planes
        return nn.Sequential(*layers)
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out); out = self.layer2(out); out = self.layer3(out)
        out = self.avgpool(out).flatten(1)
        return self.fc(out)

def resnet32(num_classes=100):
    return ResNetCIFAR((5,5,5), num_classes)


In [5]:
class DropPath(nn.Module):
    """Stochastic depth: randomly drops the residual branch per-sample during training."""
    def __init__(self, drop_prob=0.0):
        super().__init__()
        self.drop_prob = drop_prob
    def forward(self, x):
        if self.drop_prob == 0.0 or not self.training:
            return x
        keep_prob = 1 - self.drop_prob
        shape = (x.shape[0],) + (1,) * (x.ndim - 1)
        mask = keep_prob + torch.rand(shape, dtype=x.dtype, device=x.device)
        mask.floor_()
        return x.div(keep_prob) * mask

class PatchEmbed(nn.Module):
    def __init__(self, img_size=32, patch_size=4, in_ch=3, embed_dim=192):
        super().__init__()
        self.n_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_ch, embed_dim, kernel_size=patch_size, stride=patch_size)
    def forward(self, x):
        x = self.proj(x)
        return x.flatten(2).transpose(1, 2)

class LayerScaleBlock(nn.Module):
    """Pre-norm transformer block with LayerScale on both sub-branches, DeiT-III style."""
    def __init__(self, dim, n_heads, mlp_ratio=4.0, drop=0.1, drop_path=0.0, layerscale_init=1e-4):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, n_heads, dropout=drop, batch_first=True)
        self.ls1 = nn.Parameter(layerscale_init * torch.ones(dim))
        self.drop_path1 = DropPath(drop_path)

        self.norm2 = nn.LayerNorm(dim)
        hidden = int(dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(dim, hidden), nn.GELU(), nn.Dropout(drop),
            nn.Linear(hidden, dim), nn.Dropout(drop))
        self.ls2 = nn.Parameter(layerscale_init * torch.ones(dim))
        self.drop_path2 = DropPath(drop_path)

    def forward(self, x):
        h = self.norm1(x)
        attn_out, _ = self.attn(h, h, h, need_weights=False)
        x = x + self.drop_path1(self.ls1 * attn_out)
        x = x + self.drop_path2(self.ls2 * self.mlp(self.norm2(x)))
        return x

class DeiT3(nn.Module):
    def __init__(self, img_size=32, patch_size=4, in_ch=3, num_classes=100,
                 embed_dim=192, depth=8, n_heads=3, mlp_ratio=4.0, drop=0.1,
                 drop_path_max=0.1, layerscale_init=1e-4):
        super().__init__()
        self.patch_embed = PatchEmbed(img_size, patch_size, in_ch, embed_dim)
        n_patches = self.patch_embed.n_patches
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, n_patches + 1, embed_dim))
        self.pos_drop = nn.Dropout(drop)

        dpr = torch.linspace(0, drop_path_max, depth).tolist()  # linearly increasing stochastic depth
        self.blocks = nn.ModuleList([
            LayerScaleBlock(embed_dim, n_heads, mlp_ratio, drop, dpr[i], layerscale_init)
            for i in range(depth)
        ])
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)

        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)

    def forward(self, x):
        B = x.shape[0]
        x = self.patch_embed(x)
        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1) + self.pos_embed
        x = self.pos_drop(x)
        for blk in self.blocks:
            x = blk(x)
        x = self.norm(x)
        return self.head(x[:, 0])

def deit3_tiny(num_classes=100):
    # depth<18 -> layerscale_init=1e-4 per the DeiT-III paper's depth-dependent init
    return DeiT3(num_classes=num_classes, embed_dim=192, depth=8, n_heads=3,
                 drop_path_max=0.1, layerscale_init=1e-4)


## 4. Model-specific transforms, training and memory-safe evaluation

In [6]:
class ThreeAugment:
    def __init__(self):
        self.ops = [T.RandomGrayscale(p=1.0), T.RandomSolarize(threshold=128, p=1.0),
                    T.GaussianBlur(kernel_size=3)]
    def __call__(self, image):
        return random.choice(self.ops)(image)

def make_loaders(data, model_name):
    cifar_cnn = data['key'].startswith('cifar') and model_name == 'cnn'
    image_size = 32 if cifar_cnn else (LARGE_CNN_IMAGE_SIZE if model_name == 'cnn' else VIT_IMAGE_SIZE)
    if data['key'] == 'cifar10_lt':
        mean, std = (0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)
    elif data['key'] == 'cifar100_lt':
        mean, std = (0.5071, 0.4865, 0.4409), (0.2673, 0.2564, 0.2762)
    else:
        mean, std = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
    crop = T.RandomCrop(32, padding=4) if cifar_cnn else T.RandomResizedCrop(image_size, scale=(0.08, 1.0))
    ops = [crop, T.RandomHorizontalFlip()]
    if model_name == 'deit3':
        ops += [ThreeAugment(), T.ColorJitter(0.3, 0.3, 0.3)]
    train_tf = T.Compose(ops + [T.ToTensor(), T.Normalize(mean, std)])
    eval_ops = [] if cifar_cnn else [T.Resize(int(image_size / 0.875)), T.CenterCrop(image_size)]
    test_tf = T.Compose(eval_ops + [T.ToTensor(), T.Normalize(mean, std)])
    train_ds = TransformedSubset(data['train'], data['train_indices'], train_tf)
    val_ds = (TransformedSubset(data['val'], np.arange(len(data['val'])), test_tf)
              if data['val'] is not None else None)
    test_ds = TransformedSubset(data['test'], np.arange(len(data['test'])), test_tf)
    generator = torch.Generator().manual_seed(SEED)
    def seed_worker(worker_id):
        seed = torch.initial_seed() % (2**32)
        np.random.seed(seed)
        random.seed(seed)
    common = {'num_workers': NUM_WORKERS, 'pin_memory': DEVICE.type == 'cuda',
              'worker_init_fn': seed_worker}
    loaders = {
        # A singleton final batch breaks CNN BatchNorm for tiny smoke fixtures.
        'train': DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                            drop_last=len(train_ds) > BATCH_SIZE,
                            generator=generator, **common),
        'val': DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, **common) if val_ds else None,
        'test': DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, **common),
    }
    return loaders, image_size

def make_model(name, num_classes, image_size, dataset_key):
    if name == 'cnn':
        if dataset_key.startswith('cifar') or SMOKE_TEST:
            return ResNetCIFAR((1, 1, 1) if SMOKE_TEST else (5, 5, 5), num_classes)
        return torchvision.models.resnet50(weights=None, num_classes=num_classes)
    kwargs = {'img_size': image_size, 'patch_size': VIT_PATCH_SIZE,
              'num_classes': num_classes, 'embed_dim': 24 if SMOKE_TEST else 192,
              'depth': 1 if SMOKE_TEST else 8, 'n_heads': 3}
    if name == 'vit':
        return ViT(**kwargs)
    if name == 'deit3':
        return DeiT3(**kwargs, drop_path_max=0.1, layerscale_init=1e-4)
    raise ValueError(name)

def classification_loss(logits, labels, mode, train=False):
    if mode == 'bce':
        return F.binary_cross_entropy_with_logits(logits, F.one_hot(labels, logits.size(1)).float())
    return F.cross_entropy(logits, labels, label_smoothing=0.1 if train and mode == 'smooth_ce' else 0.0)

def safe_group_mean(array, group):
    values = array[group]
    finite = values[np.isfinite(values)]
    return float(finite.mean()) if len(finite) else None

@torch.no_grad()
def evaluate(model, loader, data, loss_mode):
    # Stream sufficient statistics. Never retain an N x 8142 matrix of logits.
    model.eval()
    nc = data['num_classes']
    support = np.zeros(nc, dtype=np.int64)
    predicted_count = np.zeros(nc, dtype=np.int64)
    true_positive = np.zeros(nc, dtype=np.int64)
    labels_all, preds_all = [], []
    loss_sum, top5_count, total = 0.0, 0, 0
    for batch_idx, (images, labels) in enumerate(loader):
        if MAX_EVAL_BATCHES is not None and batch_idx >= MAX_EVAL_BATCHES:
            break
        images, labels = images.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
            logits = model(images)
            loss = classification_loss(logits, labels, loss_mode)
        predictions = logits.argmax(1)
        top5_count += (logits.topk(min(5, nc), dim=1).indices == labels[:, None]).any(1).sum().item()
        ys, ps = labels.cpu().numpy(), predictions.cpu().numpy()
        support += np.bincount(ys, minlength=nc)
        predicted_count += np.bincount(ps, minlength=nc)
        true_positive += np.bincount(ys[ys == ps], minlength=nc)
        labels_all.append(ys)
        preds_all.append(ps)
        total += len(ys)
        loss_sum += float(loss) * len(ys)
    if not total:
        raise RuntimeError('Evaluation received no batches.')
    per_class_acc = np.divide(true_positive, support, out=np.full(nc, np.nan), where=support > 0)
    precision = np.divide(true_positive, predicted_count, out=np.zeros(nc, dtype=float), where=predicted_count > 0)
    recall = np.divide(true_positive, support, out=np.zeros(nc, dtype=float), where=support > 0)
    denom = support + predicted_count
    f1 = np.divide(2 * true_positive, denom, out=np.zeros(nc, dtype=float), where=denom > 0)
    ys, ps = np.concatenate(labels_all), np.concatenate(preds_all)
    cm = sparse.coo_matrix((np.ones(total, dtype=np.int64), (ys, ps)), shape=(nc, nc)).tocsr()
    cm.sum_duplicates()
    return {'loss': loss_sum / total, 'top1': float(true_positive.sum() / total),
            'top5': float(top5_count / total), 'macro_f1': float(f1.mean()),
            'weighted_f1': float(np.dot(f1, support) / total),
            'precision_macro': float(precision.mean()), 'recall_macro': float(recall.mean()),
            'group_acc': {g: safe_group_mean(per_class_acc, ids) for g, ids in data['groups'].items()},
            'per_class_acc': per_class_acc, 'support': support, 'confusion_matrix': cm,
            'labels': ys, 'predictions': ps, 'evaluated_samples': total,
            'partial_evaluation': MAX_EVAL_BATCHES is not None}

def json_ready(value):
    if isinstance(value, dict):
        return {str(k): json_ready(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(v) for v in value]
    if isinstance(value, np.ndarray):
        return json_ready(value.tolist())
    if isinstance(value, np.generic):
        return json_ready(value.item())
    if isinstance(value, float) and not math.isfinite(value):
        return None
    return value

def atomic_json(path, obj):
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(json_ready(obj), indent=2, allow_nan=False))
    temporary.replace(path)

def train_and_evaluate(model_name, data):
    set_seed()
    run_dir = OUTPUT_ROOT / NOTEBOOK_RECIPE / data['tag'] / f'{model_name}_seed{SEED}'
    if SMOKE_TEST:
        run_dir = run_dir / 'smoke_test'
    run_dir.mkdir(parents=True, exist_ok=True)
    loaders, image_size = make_loaders(data, model_name)
    model = make_model(model_name, data['num_classes'], image_size, data['key']).to(DEVICE)
    loss_mode = 'bce' if model_name == 'deit3' else ('smooth_ce' if model_name == 'vit' else 'ce')
    lr, weight_decay = (0.1, 5e-4) if model_name == 'cnn' else (1e-3, 0.05)
    optimizer = (torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay, nesterov=True)
                 if model_name == 'cnn' else torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay))
    # Clamp warmup for one-epoch smoke tests and other short jobs.
    warmup_epochs = min(10, max(0, EPOCHS - 1)) if model_name != 'cnn' else 0
    def lr_factor(epoch):
        if warmup_epochs and epoch < warmup_epochs:
            return 0.01 + 0.99 * epoch / warmup_epochs
        progress = (epoch - warmup_epochs) / max(1, EPOCHS - warmup_epochs)
        return 0.5 * (1 + math.cos(math.pi * min(1.0, progress)))
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_factor)
    scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)
    history, start_epoch, best_score = [], 0, -float('inf')
    checkpoint_path, best_path = run_dir / 'last.pt', run_dir / 'best.pt'
    selection = 'validation_top1' if loaders['val'] is not None else 'final_epoch'
    signature = {'recipe': NOTEBOOK_RECIPE, 'model': model_name, 'dataset': data['tag'],
                 'num_classes': data['num_classes'], 'image_size': image_size,
                 'epochs': EPOCHS, 'seed': SEED, 'batch_size': BATCH_SIZE,
                 'smoke_test': SMOKE_TEST, 'loss': loss_mode, 'lr': lr,
                 'train_size': len(data['train_indices']), 'data_details': data['details']}
    if RESUME and checkpoint_path.exists():
        # Only load checkpoints created by this notebook in its writable output directory.
        checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
        if checkpoint['signature'] != signature:
            raise ValueError(f'Checkpoint settings differ: {checkpoint_path}. Change OUTPUT_ROOT or set RESUME=False.')
        model.load_state_dict(checkpoint['model'])
        optimizer.load_state_dict(checkpoint['optimizer'])
        scheduler.load_state_dict(checkpoint['scheduler'])
        scaler.load_state_dict(checkpoint['scaler'])
        start_epoch, history, best_score = checkpoint['epoch'] + 1, checkpoint['history'], checkpoint['best_score']
        random.setstate(checkpoint['python_rng'])
        np.random.set_state(checkpoint['numpy_rng'])
        torch.set_rng_state(checkpoint['torch_rng'])
        loaders['train'].generator.set_state(checkpoint['loader_rng'])
        if torch.cuda.is_available() and checkpoint.get('cuda_rng') is not None:
            torch.cuda.set_rng_state_all(checkpoint['cuda_rng'])
        print(f'Resuming {data["tag"]}/{model_name} at epoch {start_epoch + 1}')
    atomic_json(run_dir / 'run_config.json', {**signature, 'selection': selection,
                                             'device': str(DEVICE), 'counts': data['counts'],
                                             'groups': data['groups']})
    np.save(run_dir / 'train_indices.npy', data['train_indices'])
    for epoch in range(start_epoch, EPOCHS):
        begin = time.time()
        model.train()
        total_loss, seen = 0.0, 0
        for batch_idx, (images, labels) in enumerate(loaders['train']):
            if MAX_TRAIN_BATCHES is not None and batch_idx >= MAX_TRAIN_BATCHES:
                break
            images, labels = images.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
                loss = classification_loss(model(images), labels, loss_mode, train=True)
            if not torch.isfinite(loss):
                raise FloatingPointError(f'Non-finite loss in {data["tag"]}/{model_name}, epoch {epoch+1}')
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            total_loss += float(loss.detach()) * len(labels)
            seen += len(labels)
        if not seen:
            raise RuntimeError('Training received no batches. Reduce BATCH_SIZE or check the split.')
        scheduler.step()
        row = {'epoch': epoch + 1, 'train_loss': total_loss / seen, 'lr': optimizer.param_groups[0]['lr']}
        if loaders['val'] is not None:
            val = evaluate(model, loaders['val'], data, loss_mode)
            row.update(val_loss=val['loss'], val_top1=val['top1'], val_top5=val['top5'], val_macro_f1=val['macro_f1'])
            if val['top1'] > best_score:
                best_score = val['top1']
                torch.save(model.state_dict(), best_path)
        row['seconds'] = time.time() - begin
        history.append(row)
        print(f'{data["tag"]}/{model_name} epoch {epoch+1}/{EPOCHS}: {row}', flush=True)
        atomic_json(run_dir / 'history.json', history)
        if (epoch + 1) % CHECKPOINT_EVERY == 0 or epoch + 1 == EPOCHS:
            checkpoint = {'signature': signature, 'epoch': epoch, 'model': model.state_dict(),
                          'optimizer': optimizer.state_dict(), 'scheduler': scheduler.state_dict(),
                          'scaler': scaler.state_dict(), 'history': history, 'best_score': best_score,
                          'python_rng': random.getstate(), 'numpy_rng': np.random.get_state(),
                          'torch_rng': torch.get_rng_state(),
                          'loader_rng': loaders['train'].generator.get_state(),
                          'cuda_rng': torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None}
            temporary = checkpoint_path.with_suffix('.tmp')
            torch.save(checkpoint, temporary)
            temporary.replace(checkpoint_path)
            del checkpoint
    if selection == 'validation_top1':
        if not best_path.exists():
            raise RuntimeError(f'Missing validation-selected checkpoint: {best_path}')
        model.load_state_dict(torch.load(best_path, map_location=DEVICE, weights_only=True))
    else:
        torch.save(model.state_dict(), run_dir / 'final.pt')
    # The final evaluation split never drives checkpoint selection.
    final = evaluate(model, loaders['test'], data, loss_mode)
    scalar_metrics = {k: v for k, v in final.items()
                      if k not in {'per_class_acc', 'support', 'confusion_matrix', 'labels', 'predictions'}}
    atomic_json(run_dir / 'final_metrics.json', scalar_metrics)
    np.save(run_dir / 'per_class_acc.npy', final['per_class_acc'])
    np.save(run_dir / 'class_support.npy', final['support'])
    np.savez_compressed(run_dir / 'predictions.npz', labels=final['labels'], predictions=final['predictions'])
    sparse.save_npz(run_dir / 'confusion_matrix_sparse.npz', final['confusion_matrix'])
    if data['num_classes'] <= 100:
        np.save(run_dir / 'confusion_matrix.npy', final['confusion_matrix'].toarray())
    group_of = {int(c): group for group, ids in data['groups'].items() for c in ids}
    pd.DataFrame({'class_id': np.arange(data['num_classes']), 'train_count': data['counts'],
                  'eval_count': final['support'], 'accuracy': final['per_class_acc'],
                  'group': [group_of[i] for i in range(data['num_classes'])]}).to_csv(run_dir / 'per_class_metrics.csv', index=False)
    result = {'dataset': data['key'], 'experiment': data['tag'], 'model': model_name, 'seed': SEED,
              'epochs': EPOCHS, 'selection': selection, 'run_dir': str(run_dir),
              **{k: v for k, v in scalar_metrics.items() if k != 'group_acc'},
              **{f'{group}_acc': value for group, value in final['group_acc'].items()},
              'smoke_test': SMOKE_TEST}
    del model, optimizer, scheduler, scaler, loaders, final
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return result


## 5. Run the five selected settings

Each experiment writes its own checkpoints and metrics. iNaturalist-2018 is excluded from the default registry.

In [7]:
all_results = []
for experiment in ACTIVE_EXPERIMENTS:
    data = prepare_experiment(experiment)
    print(f'\n=== {data["tag"]}: {len(data["train_indices"]):,} training images ===', flush=True)
    for model_name in MODEL_NAMES:
        result = train_and_evaluate(model_name, data)
        all_results.append(result)
        # Keep a usable summary even if a later run is interrupted.
        pd.DataFrame(all_results).to_csv(OUTPUT_ROOT / f'{NOTEBOOK_RECIPE}_all_metrics.csv', index=False)
    del data
    gc.collect()

results_df = pd.DataFrame(all_results)
display(results_df.drop(columns=['run_dir']))
print('Results directory:', OUTPUT_ROOT)
if SMOKE_TEST:
    print('SMOKE TEST ONLY: reduced models and partial batches. These numbers are not benchmark results.')



=== cifar10_lt_if100: 12,406 training images ===


cifar10_lt_if100/deit3 epoch 1/100: {'epoch': 1, 'train_loss': 0.3979774588439131, 'lr': 0.000109, 'seconds': 27.93099617958069}


cifar10_lt_if100/deit3 epoch 2/100: {'epoch': 2, 'train_loss': 0.24869299127956745, 'lr': 0.00020800000000000001, 'seconds': 25.16441512107849}


cifar10_lt_if100/deit3 epoch 3/100: {'epoch': 3, 'train_loss': 0.23799581956986937, 'lr': 0.000307, 'seconds': 25.827633142471313}


cifar10_lt_if100/deit3 epoch 4/100: {'epoch': 4, 'train_loss': 0.23095171053174865, 'lr': 0.00040600000000000006, 'seconds': 25.774911880493164}


...

imagenet_lt/deit3 epoch 35/100: {'epoch': 35, 'train_loss': 0.005646731956599496, 'lr': 0.0008213938048432696, 'seconds': 882.3218026161194}


imagenet_lt/deit3 epoch 36/100: {'epoch': 36, 'train_loss': 0.005626555908090585, 'lr': 0.0008078307376628291, 'seconds': 834.2362241744995}


imagenet_lt/deit3 epoch 37/100: {'epoch': 37, 'train_loss': 0.0056041338254610145, 'lr': 0.0007938926261462366, 'seconds': 796.9276061058044}


imagenet_lt/deit3 epoch 38/100: {'epoch': 38, 'train_loss': 0.005581627686338201, 'lr': 0.0007795964517353734, 'seconds': 865.8629624843597}


imagenet_lt/deit3 epoch 39/100: {'epoch': 39, 'train_loss': 0.005554984047628157, 'lr': 0.0007649596321166025, 'seconds': 906.6146128177643}


## 6. Comparisons and saved artifacts

In [ ]:
# Per-experiment plots and run histories. Large confusion matrices stay sparse on disk.
for experiment_tag, rows in results_df.groupby('experiment', sort=False):
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    for _, row in rows.iterrows():
        run_dir = Path(row['run_dir'])
        history = pd.read_json(run_dir / 'history.json')
        axes[0].plot(history['epoch'], history['train_loss'], label=row['model'])
        if 'val_top1' in history:
            axes[1].plot(history['epoch'], history['val_top1'], label=row['model'])
    axes[0].set(title=f'{experiment_tag}: training loss', xlabel='epoch', ylabel='loss')
    axes[0].legend()
    axes[1].set(title='Selection validation accuracy', xlabel='epoch', ylabel='top-1')
    if axes[1].lines:
        axes[1].legend()
    else:
        axes[1].text(0.5, 0.5, 'No separate selection validation split\nFinal-epoch checkpoint used',
                     ha='center', va='center', transform=axes[1].transAxes)
    groups = ['many_acc', 'medium_acc', 'few_acc']
    width = 0.8 / len(rows)
    for i, (_, row) in enumerate(rows.iterrows()):
        values = [np.nan if pd.isna(row[g]) else row[g] for g in groups]
        axes[2].bar(np.arange(3) + (i - (len(rows)-1)/2) * width, values, width, label=row['model'])
    axes[2].set_xticks(np.arange(3), ['head', 'mid', 'tail'])
    axes[2].set(title='Final evaluation by class group', ylabel='mean per-class accuracy', ylim=(0, 1))
    axes[2].legend()
    fig.tight_layout()
    fig.savefig(OUTPUT_ROOT / f'{NOTEBOOK_RECIPE}_{experiment_tag}_comparison.png', dpi=150)
    plt.show()
    plt.close(fig)

print('Saved: checkpoints, run configuration, sampled indices, histories, metrics, per-class CSVs,')
print('predictions and sparse confusion matrices. Accuracy values use the range 0 to 1.')
